## Topic: Structure Output 

### Agenda:
- 1. What is Structured Output?

- 2. Why Do We Need Structured Output?
    - The Problem with Normal LLM Output
    - Solution: Structured Output

- 3. Ways to get Structured Output
    - 3.1: 
    - 3.2: 
    - 3.3:

- 4. Key Takeaway

### 1. What is Structure Output?

- Definition:
    - Structured output means forcing or guiding an LLM to return information in a predefined structure or schema instead of arbitrary text or  free-form natural language text.

- Normal (Unstructured) output:
    - Textual output is an Unstructured output.

In [ ]:
# Normal Output(Unstructured) vs Structured Output
""" 
    WITHOUT Structured Output:
    ──────────────────────────
        - User(Prompt): "Extract info about Kz, age 30, software engineer at Google."

        - LLM Response (raw text):
            "Sure! Based on the information provided, Kz is a 30-year-old 
            software engineer who currently works at Google. He seems to be 
            a tech professional..."

        
        - How does our code extract the name, age, and company from this paragraph?
        Regex? String splitting? Fragile and unreliable!


    WITH Structured Output:
    ───────────────────────
        User: "Extract info about Kz, age 30, software engineer at Google."

        LLM Response (structured):
        {
            "name": "Kz",
            "age": 30,
            "job_title": "Software Engineer",
            "company": "Google"
        }

- our code can directly use this as a Python dictionary!

- No parsing needed. Type-safe. Reliable.


"""

In [ ]:
# Example:
                # WITHOUT Structured Output
                # --------------------------------
response = llm.invoke("""
Extract the candidate's name, years of experience, and skills from this resume:

"Kz, 8 years of experience in Python, TensorFlow, and AWS. 
Currently a Senior Data Scientist at Google."
""")

print(response.content)
# "The candidate is Kz. He has 8 years of experience. 
#  His skills include Python, TensorFlow, and AWS. he currently 
#  works as a Senior Data Scientist at Google."

# Now what? How do you extract the data?
# Option 1: Regex (fragile)
import re
name = re.search(r"is (\w+ \w+)", response.content).group(1)  # Breaks easily!

# Option 2: String splitting (brittle)
# What if the LLM says "Her name is Sarah" vs "The candidate is Sarah"?

# Option 3: Ask the LLM again (expensive, slow)
# Double the API cost!

### 2. Why Do We Need Structured Output?
- The Core Issues:
    - 1. Unpredictable format

    - 2. Type ambiguity

    - 3. No validation



In [ ]:
"""   
┌─────────────────────────────────────────────────────────┐
│           WHY STRUCTURED OUTPUT IS CRITICAL             │
├─────────────────────┬───────────────────────────────────┤
│  Database Storage   │ You can't INSERT a paragraph into │
│                     │ a SQL table. You need columns.    │
├─────────────────────┼───────────────────────────────────┤
│  API Responses      │ Your frontend expects JSON, not   │
│                     │ an essay from the LLM.            │
├─────────────────────┼───────────────────────────────────┤
│  Data Pipelines     │ ETL processes need typed fields   │
│                     │ (int, float, list, bool).         │ # Extract, Transform, and Load
├─────────────────────┼───────────────────────────────────┤
│  Decision Making    │ if sentiment == "Negative":       │
│                     │   trigger_alert()                 │
│                     │ ← Needs exact string, not "I      │
│                     │   think this might be negative"   │
├─────────────────────┼───────────────────────────────────┤
│  Cost Efficiency    │ Structured = fewer tokens =       │
│                     │ lower API costs                   │
└─────────────────────┴───────────────────────────────────┘


"""

### 3. 

- Benefit :
    - 1. Data Extraction
    - 2. API Building
    - 3. Build Agents
    
    - 4. To make LLM output easier to integrate with Python code, APIs, databases, and other software systems. 

    - 5. To make LLM responses predictable and consistent, so applications can reliably process the model's output using predefined fields and data types.

    - 6. To reduce formatting inconsistencies and ensure the response follows a predefined schema with specific fields and data types.



#### Key Note of Structure output: 
- Initially the LLMs output is unstructured, therefore we can't communicate or connection or integrated with the other system(like, database, API ).

- we provide an data format to get an structure output of LLMs. therefore, the LLMs are output are integrate the other system.

### 3. Ways to get Structured Output

- There are two type of LLMs:
    - 1. automatically generate the structure output.
        - example: GPT 
        - use : with_structure_output() method

    - 2. can't automatically generate the structure output
        - example: 
        - use: output_parsers class

#### 1. automatically generate the structure output.

- with_structure_output() method
    - inside the with_structure_output() method we define the Structure output formats

    - Three Common Structured Output Formats
        - 1. TypedDict

        - 2. Pydantic

        - 3. JSON Schema / dictionary schema


    - example:
        - TypedDict -> define the Structure output formats 
        - with_structure_output(TypedDict)

In [ ]:
# Example 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal

load_dotenv()

model = ChatOpenAI()

# schema
class Review(TypedDict):

    key_themes: Annotated[list[str], "Write down all the key themes discussed in the review in a list"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], "Return sentiment of the review either negative, positive or neutral"]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer"]
    

structured_model = model.with_structured_output(Review)


# prompt
user_prompt = """I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
"""

result = structured_model.invoke(user_prompt)

print(result['name'])

### 4. Key Takeaway
- In LangChain, structured output refers to the practice of having language models return
responses in a well-defined data format (for example, JSON), rather than free-form text. 

- This makes the model output easier to parse and work with programmatically.
